# Парсинг текстов

In [1]:
%%capture
!pip install selenium -q
!apt-get update -q
!apt-get install -y chromium-browser chromium-chromedriver -q
!pip install google_colab_selenium -q
!pip install corus -q

In [2]:
import re
import requests
import time
import datetime
import pandas as pd
import warnings

from tqdm import tqdm
from bs4 import BeautifulSoup
from selenium import webdriver
from dataclasses import dataclass
from datetime import datetime, timedelta
from IPython import display

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import google_colab_selenium as gs
driver = gs.Chrome()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

KeyboardInterrupt: 

### РИА Новости

In [ ]:
@dataclass
class Article:
    id: str = None
    url: str = None
    title: str = None
    subtitle: str = None
    content: str = None
    datetime: str = None

In [ ]:
SLEEP = 2
DEPTH = 30
BASE_URL = "https://ria.ru/"
TOPICS = ["economy", "society", "science", "sport", "tourism"]

Функция для сбора данных страниц со статьями

In [ ]:
def get_pages():

    """Load and scroll pages"""

    items, topics_order = [], []

    for topic in TOPICS:
        try:
            old_size = len(items)
            URL = BASE_URL + topic
            driver.get(URL)
            time.sleep(SLEEP)

            # push to list 20 next articles
            driver.execute_script(
                "document.getElementsByClassName('list-more')[0].click()"
            )
            time.sleep(1)

            # scroll page to automatically load more articles
            for i in tqdm(range(DEPTH), leave=False):
                try:
                    driver.execute_script(
                        f"window.scrollTo(0, document.body.scrollHeight - 1200)"
                    )
                    time.sleep(1)
                except:
                    pass

            # find all pages
            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")
            scope = soup.find(
                "div", {"class": "list", "itemtype": "http://schema.org/ItemList"}
            )
            items += scope.find_all("div", {"class": "list-item"})

            # number of pages can not be multiple of deepth*20
            # that's why we count topics_order dynamically
            new_size = len(items)
            if new_size > old_size:
                topics_order.extend([topic] * (new_size - old_size))
        except:
            pass

    return items, topics_order

Функция для парсинга контента со страниц

In [ ]:
def parse_page(page):
    """Extract from page desired fields"""

    # Create article data class object
    article = Article()

    # article url
    article.url = page.find("a", {"class": "list-item__image"})["href"]

    # article id
    s = re.findall(r"\d+.html", article.url)[0]
    article.id = s[: s.find(".")]

    # load page
    driver.get(article.url)
    time.sleep(SLEEP)
    html = driver.page_source

    # article source
    source = article.url[8 : article.url.find(".")]

    # article object
    soup = BeautifulSoup(html, "html.parser")
    obj = soup.find(
        "div",
        {
            "class": lambda x: x and (x.find(f"article m-article m-{source}") > -1),
            "data-article-id": article.id,
        },
    )

    if not obj:
        obj = soup.find(
            "div",
            {
                "class": lambda x: x and (x.find(f"article m-video m-{source}") > -1),
                "data-article-id": article.id,
            },
        )

    # title
    title = obj.find("div", {"class": "article__title"})
    title_2 = obj.find("h1", {"class": "article__title"})

    if title:
        article.title = title.text
    else:
        article.title = title_2.text if title_2 else ""

    # subtitle
    subtitle = obj.find("h1", {"class": "article__second-title"})
    article.subtitle = subtitle.text if subtitle else ""

    # content
    article.content = obj.find(
        "div", {"class": "article__body js-mediator-article mia-analytics"}
    ).text

    # datetime
    article.datetime = obj.find("div", {"class": "article__info-date"}).find("a").text

    return article

In [ ]:
# pages, topics_order = get_pages()
# len(pages)

In [ ]:
# data, topics_order_fixed = [], []

# for num, page in enumerate(tqdm(pages)):
#     try:
#         res = parse_page(page)

#         data.append(res)
#         topics_order_fixed.append(topics_order[num])
#     except:
#         pass

In [ ]:
# df = pd.DataFrame(data=data)

# df.info()

In [ ]:
# df["topic"] = topics_order_fixed

In [ ]:
# df.head()

In [ ]:
# df.to_csv('ria_news.csv')

### Lenta.ru

In [ ]:
class lentaRu_parser:
    def __init__(self):
        pass

    def _get_url(self, param_dict: dict) -> str:
        """
        Возвращает URL для запроса json таблицы со статьями
        """
        hasType = int(param_dict['type']) != 0
        hasBloc = int(param_dict['bloc']) != 0

        url = (
            'https://lenta.ru/search/v2/process?'
            + 'from={}&'.format(param_dict['from'])
            + 'size={}&'.format(param_dict['size'])
            + 'sort={}&'.format(param_dict['sort'])
            + 'title_only={}&'.format(param_dict['title_only'])
            + 'domain={}&'.format(param_dict['domain'])
            + 'modified%2Cformat=yyyy-MM-dd&'
        )

        # Добавляем условные параметры только если они нужны
        if hasType:
            url += 'type={}&'.format(param_dict['type'])
        if hasBloc:
            url += 'bloc={}&'.format(param_dict['bloc'])

        url += (
            'modified%2Cfrom={}&'.format(param_dict['dateFrom'])
            + 'modified%2Cto={}&'.format(param_dict['dateTo'])
            + 'query={}'.format(param_dict['query'])
        )

        return url

    def _get_search_table(self, param_dict: dict) -> pd.DataFrame:
        """
        Возвращает pd.DataFrame со списком статей
        """
        url = self._get_url(param_dict)
        r = requests.get(url)
        r.raise_for_status()  # полезно для явной обработки ошибок
        search_table = pd.DataFrame(r.json()['matches'])
        return search_table

    def get_articles(
        self,
        param_dict,
        time_step=37,
        save_every=5
    ) -> pd.DataFrame:
        """
        Функция для скачивания статей интервалами через каждые time_step дней
        Делает сохранение таблицы через каждые save_every * time_step дней
        """
        param_copy = param_dict.copy()
        time_step = timedelta(days=time_step)
        dateFrom = datetime.strptime(param_copy['dateFrom'], '%Y-%m-%d')
        dateTo = datetime.strptime(param_copy['dateTo'], '%Y-%m-%d')
        if dateFrom > dateTo:
            raise ValueError('dateFrom should be less than dateTo')

        out = pd.DataFrame()
        save_counter = 0

        while dateFrom <= dateTo:
            param_copy['dateTo'] = (dateFrom + time_step).strftime('%Y-%m-%d')
            if dateFrom + time_step > dateTo:
                param_copy['dateTo'] = dateTo.strftime('%Y-%m-%d')

            print(
                'Parsing articles from '
                + param_copy['dateFrom'] + ' to ' + param_copy['dateTo']
            )

            chunk_df = self._get_search_table(param_copy)

            out = pd.concat([out, chunk_df], ignore_index=True)

            dateFrom += time_step + timedelta(days=1)
            param_copy['dateFrom'] = dateFrom.strftime('%Y-%m-%d')
            save_counter += 1

            if save_counter == save_every:
                display.clear_output(wait=True)
                out.to_excel("/tmp/checkpoint_table.xlsx", index=False)
                print('Checkpoint saved!')
                save_counter = 0

        print('Finish')

        out = out.drop(columns=['modified', 'lastmodtime', 'type',
                                'domain', 'status', 'part',
                                'tags', 'image_url', 'rightcol'])
        out.rename(columns={'snippet': 'subtitle', 'docid': 'id', 'text': 'content', 'pubdate': 'datetime'},
                   inplace=True)

        return out

In [ ]:
parser = lentaRu_parser()

In [ ]:
query = ''
offset = 0
size = 1000
sort = "3"
title_only = "0"
domain = "1"
material = "0"
bloc = "0" # topic = тематика новости
dateFrom = '2023-01-01'
dateTo = "2026-01-14"

lenta_blocks = {'Общество': 1,
                'Экономика': 4,
                'Силовые структуры': 37,
                'Бывший СССР': 3,
                'Спорт': 8,
                'Забота о себе': 87,
                'Строительство': 0,
                'Туризм/Путешествия': 48,
                'Наука и техника': 5,}

In [ ]:
lenta_df = pd.DataFrame(columns=['id', 'url', 'title', 'bloc', 'datetime', 'content', 'subtitle'])

for query in lenta_blocks.keys():

    param_dict = {'query'     : query.lower(),
                  'from'      : str(offset),
                  'size'      : str(size),
                  'dateFrom'  : dateFrom,
                  'dateTo'    : dateTo,
                  'sort'      : sort,
                  'title_only': title_only,
                  'type'      : material,
                  'bloc'      : bloc,
                  'domain'    : domain}

    tbl = parser.get_articles(param_dict=param_dict,
                           time_step=37,
                           save_every=5)


    lenta_df = pd.concat([lenta_df, tbl], ignore_index=True)

len(lenta_df)

Checkpoint saved!
Finish


49456

In [ ]:
lenta_df.to_csv('lenta_news_upd.csv', index=False)

#  Первичная обработка и разметка тем

### РИА Новости

In [ ]:
ria_df = pd.read_csv('ria_news.csv').drop(columns=['Unnamed: 0'])

In [ ]:
ria_df['topic'].value_counts()

,count
topic,
science,624
society,379
economy,375


In [ ]:
topic_mapping = {
    'science': 8,
    'society': 0,
    'economy': 1
}

ria_df['topic'] = ria_df['topic'].map(topic_mapping)

In [ ]:
ria_df['content'].sample(5)

,content
1252,"МОСКВА, 19 сен – РИА Новости. Куликовская битв..."
1089,"МОСКВА, 18 окт - РИА Новости. Значительное уве..."
49,"МОСКВА, 16 дек - РИА Новости. Иск Банка России..."
1166,"МОСКВА, 7 окт — РИА Новости. Нобелевскую преми..."
1100,"МОСКВА, 17 окт – РИА Новости. Два ИТ-решения ""..."


Удалим шапку

In [ ]:
pattern = r'^[А-ЯЁ]+(?:-[А-ЯЁ]+)?,\s*\d{1,2}\s+[а-яё]+\.?\s*[–\-—]\s*РИА Новости\.?\s*'
ria_df['content'] = ria_df['content'].apply(lambda x: re.sub(pattern, '', x))
ria_df['content'].sample(5)

,content
1372,Грузовой космический корабль Dragon американск...
42,Еврокомиссия в рамках опубликованного пакета м...
463,"В салонах автобусов ""Мострансавто"" начали звуч..."
1359,"Противоударный учебный беспилотник, оснащенный..."
886,"Новый материал, который работает как ловушка д..."


In [ ]:
ria_df['content'].str[-20:].sample(5)

,content
973,науки и технологий.
1266,18:43\n\n\nПоделиться\n\n
1352,"ств"", – заключил он."
438,21:43\n\n\nПоделиться\n\n
1162,07:32\n\n\nПоделиться\n\n


In [ ]:
ria_df['content'] = ria_df['content'].str.replace('Поделиться', '', regex=False)

In [ ]:
ria_df.shape

(1378, 7)

### Lenta.ru

In [41]:
lenta_df = pd.read_csv('lenta_news_upd.csv')

Замапим номера тем API Ленты и нашего соревнования

In [42]:
mapping_dict = {1: 0, 3: 3, 4: 1, 5: 8, 8: 4, 37: 2, 48: 7, 87: 5}

lenta_df['topic'] = lenta_df['bloc'].map(mapping_dict)

In [43]:
lenta_df['topic'].isna().sum()

np.int64(12418)

Так как у ленты нет отдельной рубрики под тему Строительство, разметим её по ключевым словам

In [ ]:
AUTH_KEY = 'NjI2NTU4MWEtOGE4NC00YmFkLTljZTItNDIyOGExYjVmOTgwOmQyOGViNDIzLWZlNGItNDdlNi05MzYzLTllZDA4YTYxMmFkMw=='

In [ ]:
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole

giga = GigaChat(
    model="GigaChat-2",
    credentials="ключ_авторизации",
    scope="GIGACHAT_API_PERS",
)


categories_list = "\n".join([f"- {cat}" for cat in lenta_blocks.keys()])

def classify_text(text):
    if pd.isna(text) or text == "":
        return "Ошибка"

    payload = Chat(
        messages=[
            Messages(
                role=MessagesRole.SYSTEM,
                content=f"""Ты - профессиональный классификатор текстов.
Твоя задача – определить тематику текста по одной из следующих категорий:
{categories_list}

## Правила:
- Анализируй содержание текста внимательно
- Выбирай наиболее подходящую категорию
- Твой ответ должен содержать ТОЛЬКО название выбранной категории из списка выше
- Не добавляй никаких пояснений или комментариев

## Формат ответа:
Название категории (без прочих слов)"""
            ),
            Messages(
                role=MessagesRole.USER,
                content=str(text)
            ),
        ],
        temperature=0.1
    )

    try:
        response = giga.chat(payload)
        category = response.choices[0].message.content.strip()

        if category in lenta_blocks:
            return category
        else:
            return 'Неожиданная категория'

    except Exception as e:
        print(f"Ошибка при классификации: {e}")
        return "Ошибка"

In [ ]:
lenta_df['topic'] = lenta_df.apply(
    lambda row: classify_text(row['content']) if pd.isna(row['topic'])
                  else row['topic'],
    axis=1
)
lenta_df.to_csv('classified_texts.csv', index=False)

In [44]:
construction_keywords = [
    'стройка',
    'строительные работы',

    'застройщик',
    'застройка',
    'стройплощадка',
    'строительная площадка',
    'генподрядчик',

    'возведение здания',
    'возведение объекта',
    'сдача объекта',
    'ввод в эксплуатацию',

    'строительные материалы',
    'стройматериалы',
    'цемент',
    'бетон',
    'кирпич',

    'строительная техника',
    'строительный кран',
    'экскаватор',
    'бульдозер',

    'строитель',
    'прораб',
    'инженер-строитель'
]

pattern = '|'.join(construction_keywords)

In [45]:
mask = lenta_df['topic'].isna()

lenta_df.loc[mask, 'topic'] = lenta_df.loc[mask, 'content'].str.contains(
    pattern,
    case=False,
    regex=True,
    na=False
).map({True: 6, False: None})

In [46]:
lenta_df = lenta_df[pd.notna(lenta_df['topic'])]

In [47]:
lenta_df['topic'].value_counts()

,count
topic,
1.0,13015
0.0,8778
4.0,4969
3.0,4653
6.0,3425
2.0,2604
8.0,1195
5.0,1175
7.0,649


Уберем статьи, не попавшие ни под одну целевую тему

In [ ]:
lenta_df = lenta_df.dropna(subset=['topic'])
lenta_df = lenta_df.drop(columns=['bloc'])

In [ ]:
lenta_df.shape

(39783, 7)

### Lenta.ru (Corus)

In [54]:
from corus import load_lenta2

path = 'lenta-ru-news.csv.bz2'
records = load_lenta2(path)

chunk_size = 10000
chunks = []
temp_records = []

for i, record in enumerate(records):
    temp_records.append({
        'url': record.url,
        'title': record.title,
        'text': record.text,
        'topic': record.topic,
        'tags': record.tags,
        'date': record.date
    })

    if (i + 1) % chunk_size == 0:
        chunks.append(pd.DataFrame(temp_records))
        print(f"Обработано {i + 1} записей")
        temp_records = []

if temp_records:
    chunks.append(pd.DataFrame(temp_records))
    print(f"Финальная обработка: {len(temp_records)} записей")

corus_df = pd.concat(chunks, ignore_index=True)

Обработано 10000 записей
Обработано 20000 записей
Обработано 30000 записей
Обработано 40000 записей
Обработано 50000 записей
Обработано 60000 записей
Обработано 70000 записей
Обработано 80000 записей
Обработано 90000 записей
Обработано 100000 записей
Обработано 110000 записей
Обработано 120000 записей
Обработано 130000 записей
Обработано 140000 записей
Обработано 150000 записей
Обработано 160000 записей
Обработано 170000 записей
Обработано 180000 записей
Обработано 190000 записей
Обработано 200000 записей
Обработано 210000 записей
Обработано 220000 записей
Обработано 230000 записей
Обработано 240000 записей
Обработано 250000 записей
Обработано 260000 записей
Обработано 270000 записей
Обработано 280000 записей
Обработано 290000 записей
Обработано 300000 записей
Обработано 310000 записей
Обработано 320000 записей
Обработано 330000 записей
Обработано 340000 записей
Обработано 350000 записей
Обработано 360000 записей
Обработано 370000 записей
Обработано 380000 записей
Обработано 390000 зап

In [55]:
corus_df = corus_df.drop(columns=['url', 'title', 'date', 'tags'])
print(f"\nИтого: {len(corus_df)} строк")
print(corus_df.info())
print(corus_df.columns.tolist())


Итого: 800975 строк
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800975 entries, 0 to 800974
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    800975 non-null  object
 1   topic   800975 non-null  object
dtypes: object(2)
memory usage: 12.2+ MB
None
['text', 'topic']


In [56]:
topic_list = ['Россия', 'Экономика', 'Спорт', 'Бывший СССР', 'Силовые структуры',
              'Путешествия', 'Наука и техника']
corus_df = corus_df[corus_df['topic'].isin(topic_list)]

In [57]:
corus_df['topic'].value_counts()

,count
topic,
Россия,160445
Экономика,79528
Спорт,64413
Бывший СССР,53402
Наука и техника,53136
Силовые структуры,19596
Путешествия,6408


In [58]:
topic_mapping = {
    'Россия': 0,
    'Экономика': 1,
    'Силовые структуры': 2,
    'Бывший СССР': 3,
    'Спорт': 4,
    'Путешествия': 7,
    'Наука и техника': 8
}

corus_df['topic'] = corus_df['topic'].map(topic_mapping)

In [60]:
corus_df = corus_df.rename(columns={'text': 'content'})

In [62]:
corus_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 436928 entries, 5 to 739175
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   content  436928 non-null  object
 1   topic    436928 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 10.0+ MB


# Итоговый датасет

In [63]:
res_df = pd.concat([corus_df, lenta_df], ignore_index=True)
res_df.shape

(477391, 8)

In [64]:
res_df['topic'].value_counts()

,count
topic,
0.0,169223
1.0,92543
4.0,69382
3.0,58055
8.0,54331
2.0,22200
7.0,7057
6.0,3425
5.0,1175


In [65]:
res_df.to_csv('corus_lenta.csv', index=False)